# CSGO EDA

Exploratory data analysis for the real CSGO archive benchmark.

This notebook reads the exported feature tables generated by `python3 scripts/export_csgo_bundle.py` and helps answer:

- what the archive contains
- how legit and cheater sessions differ
- which engineered features separate the classes best
- what the current exported benchmark looks like before modeling changes


In [1]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'csgo_generated'

required = [
    DATA / 'metadata.json',
    DATA / 'evaluation_metrics.csv',
    DATA / 'split_metrics.csv',
    DATA / 'window_features.csv',
    DATA / 'classifier_windows.csv',
    DATA / 'session_feature_table.csv',
]
missing = [path.name for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'Missing exported files: ' + ', '.join(missing) + '. Run python3 scripts/export_csgo_bundle.py first.'
    )

metadata = json.loads((DATA / 'metadata.json').read_text())
eval_metrics = pd.read_csv(DATA / 'evaluation_metrics.csv')
split_metrics = pd.read_csv(DATA / 'split_metrics.csv')
window_features = pd.read_csv(DATA / 'window_features.csv')
classifier_windows = pd.read_csv(DATA / 'classifier_windows.csv')
session_features = pd.read_csv(DATA / 'session_feature_table.csv')


In [2]:
print('Archive summary')
display(pd.DataFrame([metadata['archive_summary']]))

print('Model summary')
display(pd.DataFrame([metadata.get('model_summary', {})]))

print('Held-out metrics')
display(eval_metrics)

print('Split metrics')
display(split_metrics)


Archive summary


,legit_players_in_archive,cheater_players_in_archive,engagements_per_player,timesteps_per_engagement,raw_feature_count,window_size,window_stride,legit_players_used,cheater_players_used
0,10000,2000,30,192,5,32,8,60,30


Model summary


,window_encoder,cheat_scorer,train_once_export_pattern
0,sklearn MLPClassifier over flattened causal wi...,sklearn LogisticRegression over engineered win...,True


Held-out metrics


,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,false_positive_rate,false_negative_rate,prevalence,predicted_positive_rate,majority_baseline_accuracy,ppv_at_observed_prevalence,npv_at_observed_prevalence,positive_likelihood_ratio,negative_likelihood_ratio,tp,fp,tn,fn
0,0.652381,0.652593,0.510417,0.653333,0.651852,0.573099,0.293544,0.348148,0.346667,0.357143,0.457143,0.642857,0.510417,0.77193,1.876596,0.531818,98,94,176,52


Split metrics


,split,accuracy,balanced_accuracy,precision,recall,specificity,f1,mcc,false_positive_rate,false_negative_rate,...,predicted_positive_rate,majority_baseline_accuracy,ppv_at_observed_prevalence,npv_at_observed_prevalence,positive_likelihood_ratio,negative_likelihood_ratio,tp,fp,tn,fn
0,train,0.738095,0.772222,0.569804,0.874603,0.669841,0.690044,0.513446,0.330159,0.125397,...,0.511640,0.666667,0.569804,0.914410,2.649038,0.187204,551.0,416.0,844.0,79.0
1,validation,0.507692,0.521759,0.325243,0.558333,0.485185,0.411043,0.040235,0.514815,0.441667,...,0.528205,0.692308,0.325243,0.711957,1.084532,0.910305,67.0,139.0,131.0,53.0
2,test,0.652381,0.652593,0.510417,0.653333,0.651852,0.573099,0.293544,0.348148,0.346667,...,0.457143,0.642857,0.510417,0.771930,1.876596,0.531818,98.0,94.0,176.0,52.0


In [3]:
split_balance = (
    session_features.groupby(['split', 'label_cheat'])['session_id']
    .count()
    .reset_index(name='session_count')
)
display(split_balance)

fig = px.bar(split_balance, x='split', y='session_count', color='label_cheat', barmode='group', title='Session Count by Split and Label')
fig.show()


,split,label_cheat,session_count
0,test,0,270
1,test,1,150
2,train,0,1260
3,train,1,630
4,validation,0,270
5,validation,1,120


In [4]:
test_sessions = session_features[session_features['split'] == 'test'].copy()
feature_cols = [c for c in test_sessions.columns if c not in {'session_id', 'player_id', 'player_index', 'source_label', 'split', 'engagement_index', 'label_cheat'}]

rows = []
legit = test_sessions[test_sessions['label_cheat'] == 0]
cheat = test_sessions[test_sessions['label_cheat'] == 1]
for col in feature_cols:
    legit_vals = legit[col].astype(float)
    cheat_vals = cheat[col].astype(float)
    pooled = np.sqrt((legit_vals.var() + cheat_vals.var()) / 2.0 + 1e-9)
    effect = (cheat_vals.mean() - legit_vals.mean()) / pooled
    rows.append({
        'feature': col,
        'legit_mean': legit_vals.mean(),
        'cheat_mean': cheat_vals.mean(),
        'effect_size': effect,
        'abs_effect_size': abs(effect),
    })

effect_table = pd.DataFrame(rows).sort_values('abs_effect_size', ascending=False)
display(effect_table.head(20))


,feature,legit_mean,cheat_mean,effect_size,abs_effect_size
5,encoder_signal_mean,0.426837,0.818601,0.958508,0.958508
0,cheat_probability_mean,0.422490,0.511102,0.957350,0.957350
1,cheat_probability_max,0.734826,0.788911,0.534137,0.534137
6,encoder_signal_max,1.807303,2.025091,0.494505,0.494505
2,cheat_probability_std,0.147680,0.166847,0.464883,0.464883
22,lock_strength_rise_max,1.082184,0.387864,-0.417537,0.417537
25,fire_coupling_shift_mean,0.415281,0.338924,-0.394705,0.394705
21,lock_strength_rise_mean,0.219633,0.071964,-0.391910,0.391910
29,fire_stability_synergy_rise_mean,0.036559,0.021851,-0.359492,0.359492
43,micro_correction_drop_mean,0.326708,0.251770,-0.341528,0.341528


In [5]:
top_features = effect_table.head(8)['feature'].tolist()
melted = test_sessions.melt(id_vars=['label_cheat'], value_vars=top_features, var_name='feature', value_name='value')
fig = px.box(melted, x='feature', y='value', color='label_cheat', points=False, title='Top Session-Level Separators on the Test Split')
fig.update_layout(xaxis_tickangle=-35, height=550)
fig.show()


In [6]:
fig = px.scatter(
    test_sessions,
    x='cheat_probability_max',
    y='encoder_signal_max',
    color='label_cheat',
    hover_data=['session_id', 'player_id', 'source_label'],
    title='Session Separation: Window Probability vs Encoder Signal',
)
fig.show()

fig = px.scatter(
    test_sessions,
    x='fire_coupling_shift_max',
    y='snap_power_rise_max',
    color='label_cheat',
    hover_data=['session_id', 'player_id'],
    title='Fire Coupling vs Snap Power',
)
fig.show()


In [7]:
window_test = classifier_windows[classifier_windows['split'] == 'test'].copy()
fig = px.histogram(window_test, x='cheat_probability', color='label_cheat', nbins=60, barmode='overlay', opacity=0.55, title='Window-Level Cheat Probability on Test Windows')
fig.show()

fig = px.histogram(window_test, x='encoder_signal', color='label_cheat', nbins=60, barmode='overlay', opacity=0.55, title='Window-Level Encoder Signal on Test Windows')
fig.show()
